# 01 - Loop do agente: análise do mercado de milho

Todo agente funciona em um **loop**: o modelo decide o que fazer, uma ferramenta executa a ação, o resultado volta para o modelo e o ciclo se repete até o modelo dar a resposta final.

Neste notebook, um deep agent faz o papel de analista do mercado de milho. Ele consulta preços, clima e estoques, e no fim escreve um relatório com recomendação. Os dados são **simulados** (fictícios), para o foco ficar no loop e não nas fontes reais.

O primeiro nível do loop é o mais simples: sem memória de longo prazo, sem subagentes, só o ciclo básico de decisão e ação.

Fluxo:

```
START → modelo → tem tool_calls? → sim → ferramentas → modelo → ...
                                  → não → resposta final → END
```

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

import mlflow

from deepagents import create_deep_agent

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "loop-agente-milho"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

2026/11/17 18:18:36 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/leonardo/Documentos/github/loop_engineering_study/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/11/17 18:18:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/17 18:18:36 INFO mlflow.store.db.utils: Updating database tables


2026/11/17 18:18:38 INFO mlflow.tracking.fluent: Experiment with name 'loop-agente-milho' does not exist. Creating a new experiment.


## 2. Dados simulados

Estes dados são fictícios. Eles ficam em dicionários para o notebook funcionar sem internet e sempre com o mesmo resultado. Em um agente real, cada ferramenta chamaria uma API ou um banco de dados.

In [2]:
PRECOS_SACA = {
    "paranagua": {"praca": "Paranaguá (PR)", "preco_rs_saca": 118.50, "variacao_7_dias_pct": 1.8},
    "cascavel": {"praca": "Cascavel (PR)", "preco_rs_saca": 112.30, "variacao_7_dias_pct": 2.4},
    "rio verde": {"praca": "Rio Verde (GO)", "preco_rs_saca": 108.90, "variacao_7_dias_pct": 0.6},
    "campinas": {"praca": "Campinas (SP)", "preco_rs_saca": 120.10, "variacao_7_dias_pct": -0.9},
}

CLIMA = {
    "parana": {"regiao": "Oeste do Paraná", "chuva_ultimos_7_dias_mm": 12, "chuva_prevista_15_dias_mm": 18,
               "risco": "baixo", "observacao": "Chuva irregular no fim do mês, sem sinal de seca forte."},
    "mato grosso": {"regiao": "Mato Grosso", "chuva_ultimos_7_dias_mm": 4, "chuva_prevista_15_dias_mm": 6,
                    "risco": "alto", "observacao": "Veranico prolongado, risco de perda na segunda safra."},
    "goias": {"regiao": "Goiás", "chuva_ultimos_7_dias_mm": 7, "chuva_prevista_15_dias_mm": 10,
              "risco": "médio", "observacao": "Chuva abaixo da média para a época."},
}

ESTOQUES = {
    "safra_estimada_mil_t": 128_000,
    "consumo_estimado_mil_t": 79_000,
    "estoque_inicial_mil_t": 21_000,
    "exportacao_estimada_mil_t": 42_000,
    "estoque_final_estimado_mil_t": 28_000,
    "relacao_estoque_consumo_pct": 35.4,
    "leitura": "Estoque final abaixo da média dos últimos 5 anos.",
}

## 3. Ferramentas

Cada ferramenta é uma função com `@tool`. O **nome**, os **argumentos** e a **docstring** são o que o modelo lê para decidir quando chamar cada uma. Por isso a docstring explica o uso em português e de forma direta.

Quando a ferramenta não encontra o item pedido, ela devolve as opções válidas. Assim o modelo pode corrigir a chamada no próximo passo do loop.

In [3]:
@tool
def consultar_preco_milho(praca: str) -> str:
    """Consulta o preço spot do milho em uma praça. Praças válidas: paranagua, cascavel, rio verde, campinas."""
    chave = praca.lower().strip()
    if chave not in PRECOS_SACA:
        return json.dumps({"erro": f"Praça '{praca}' não encontrada.", "pracas_validas": list(PRECOS_SACA)}, ensure_ascii=False)
    return json.dumps(PRECOS_SACA[chave], ensure_ascii=False)


@tool
def consultar_clima(regiao: str) -> str:
    """Consulta a chuva recente, a previsão e o risco climático de uma região produtora. Regiões válidas: parana, mato grosso, goias."""
    chave = regiao.lower().strip()
    if chave not in CLIMA:
        return json.dumps({"erro": f"Região '{regiao}' não encontrada.", "regioes_validas": list(CLIMA)}, ensure_ascii=False)
    return json.dumps(CLIMA[chave], ensure_ascii=False)


@tool
def consultar_estoques() -> str:
    """Consulta o balanço de oferta e demanda do milho no Brasil: safra, consumo, exportação e estoque final."""
    return json.dumps(ESTOQUES, ensure_ascii=False)


FERRAMENTAS = [consultar_preco_milho, consultar_clima, consultar_estoques]

## 4. Criando o agente

`create_deep_agent` recebe o modelo, as ferramentas e o `system_prompt`. O prompt define o papel e a regra de trabalho: coletar os dados antes de concluir.

O deep agent já vem com ferramentas de planejamento e de arquivos. Elas continuam no loop, mas este notebook usa só as ferramentas de mercado.

In [4]:
agente = create_deep_agent(
    model=llm,
    tools=FERRAMENTAS,
    system_prompt=(
        "Você é um analista do mercado de milho. "
        "Antes de concluir, use as ferramentas para reunir os dados: preços em pelo menos duas praças, "
        "o clima de pelo menos uma região produtora e o balanço de estoques. "
        "Quando tiver os dados, escreva em português um relatório curto com: "
        "situação dos preços, risco climático, leitura dos estoques e recomendação (vender, comprar ou aguardar), "
        "com a justificativa. Lembre que os dados são simulados."
    ),
)

## 5. Acompanhando o loop

O `stream` com `stream_mode="updates"` devolve cada passo do loop assim que ele termina. A função `executar_com_log` mostra:

- **modelo** decidindo chamar uma ferramenta (`tool_calls`);
- **ferramentas** devolvendo a observação (`ToolMessage`);
- **modelo** dando a resposta final, quando não há mais chamadas.

Cada volta do modelo às ferramentas é uma iteração do loop. O número de iterações é o que o notebook conta.

In [5]:
def resumir(texto: str, limite: int = 160) -> str:
    texto = texto.replace("\n", " ")
    return texto if len(texto) <= limite else texto[:limite] + "..."


def executar_com_log(pergunta: str, limite_recursao: int = 40) -> list:
    print(f"Pergunta: {pergunta}\n")
    passo = 0
    mensagens = []
    eventos = agente.stream(
        {"messages": [HumanMessage(content=pergunta)]},
        config={"recursion_limit": limite_recursao},
        stream_mode="updates",
    )
    for evento in eventos:
        for no, atualizacao in evento.items():
            if not isinstance(atualizacao, dict):
                continue
            bloco = atualizacao.get("messages", [])
            bloco = bloco if isinstance(bloco, list) else [bloco]
            for mensagem in bloco:
                mensagens.append(mensagem)
                if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
                    passo += 1
                    for chamada in mensagem.tool_calls:
                        print(f"[{passo}] modelo decide: {chamada['name']}({chamada['args']})")
                elif isinstance(mensagem, ToolMessage):
                    print(f"    ferramenta devolve: {resumir(str(mensagem.content))}")
                elif isinstance(mensagem, AIMessage) and mensagem.content:
                    print(f"\n[fim do loop] resposta final do modelo:\n{mensagem.content}")
    print(f"\nIterações do loop com ferramentas: {passo}")
    return mensagens


mensagens = executar_com_log(
    "Como está o mercado de milho agora e qual deve ser a recomendação para um produtor do Paraná?"
)

Pergunta: Como está o mercado de milho agora e qual deve ser a recomendação para um produtor do Paraná?



[1] modelo decide: consultar_preco_milho({'praca': 'paranagua'})
[1] modelo decide: consultar_preco_milho({'praca': 'cascavel'})
[1] modelo decide: consultar_clima({'regiao': 'parana'})
[1] modelo decide: consultar_estoques({})
    ferramenta devolve: {"regiao": "Oeste do Paraná", "chuva_ultimos_7_dias_mm": 12, "chuva_prevista_15_dias_mm": 18, "risco": "baixo", "observacao": "Chuva irregular no fim do mês, se...
    ferramenta devolve: {"praca": "Paranaguá (PR)", "preco_rs_saca": 118.5, "variacao_7_dias_pct": 1.8}
    ferramenta devolve: {"safra_estimada_mil_t": 128000, "consumo_estimado_mil_t": 79000, "estoque_inicial_mil_t": 21000, "exportacao_estimada_mil_t": 42000, "estoque_final_estimado_mi...
    ferramenta devolve: {"praca": "Cascavel (PR)", "preco_rs_saca": 112.3, "variacao_7_dias_pct": 2.4}



[fim do loop] resposta final do modelo:
Resumo curto (dados simulados)

1) Situação dos preços
- Paranaguá (PR): R$ 118,5 / saca (variação 7 dias: +1,8%)
- Cascavel (PR): R$ 112,3 / saca (variação 7 dias: +2,4%)
- Interpretação: preços no Paraná estão firmes e com leve alta semanal; diferença entre praças sugere basis local favorável em Paranaguá.

2) Risco climático (Oeste do Paraná)
- Chuva últimos 7 dias: 12 mm; previsão 15 dias: 18 mm.
- Avaliação de risco: baixo — chuvas moderadas e sem sinal de seca forte, com possibilidade de chuva irregular no fim do mês.
- Impacto: condições não apontam para risco imediato de perda de safra ou secas que derrubem preços; colheita e secagem devem seguir sem grandes surpresas.

3) Leitura dos estoques / balanço (Brasil)
- Safra estimada: 128.000 mil t
- Consumo estimado: 79.000 mil t
- Exportação estimada: 42.000 mil t
- Estoque final estimado: 28.000 mil t
- Relação estoque/consumo: 35,4%
- Interpretação: estoque final abaixo da média dos últim

## 6. Conferindo o loop

Com as mensagens guardadas, dá para contar as chamadas de ferramenta e ver a ordem em que cada uma foi feita. Isso ajuda a checar se o agente reuniu os dados antes de concluir.

In [6]:
ordem = []
for mensagem in mensagens:
    if isinstance(mensagem, AIMessage):
        ordem.extend(chamada["name"] for chamada in mensagem.tool_calls)

print("Ordem das ferramentas:", " → ".join(ordem) if ordem else "nenhuma")
print("Consultou preço:", "consultar_preco_milho" in ordem)
print("Consultou clima:", "consultar_clima" in ordem)
print("Consultou estoques:", "consultar_estoques" in ordem)

Ordem das ferramentas: consultar_preco_milho → consultar_preco_milho → consultar_clima → consultar_estoques
Consultou preço: True
Consultou clima: True
Consultou estoques: True


## 7. Limite de iterações

Um loop sem limite pode não parar: se o modelo ficar chamando ferramentas sem concluir, o custo cresce a cada volta. O `recursion_limit` define o número máximo de passos do grafo. Quando o limite é atingido, o LangGraph lança `GraphRecursionError`.

Esta célula usa um limite baixo de propósito, para mostrar o erro. Em uso real, o limite fica alto o suficiente para a tarefa e baixo o bastante para limitar o custo.

In [7]:
from langgraph.errors import GraphRecursionError

try:
    agente.invoke(
        {"messages": [HumanMessage(content="Analise o mercado de milho nas quatro praças e nas três regiões.")]},
        config={"recursion_limit": 4},
    )
    print("Terminou dentro do limite.")
except GraphRecursionError as erro:
    print("Loop interrompido pelo limite de passos:")
    print(f"{type(erro).__name__}: {erro}")

Loop interrompido pelo limite de passos:
GraphRecursionError: Recursion limit of 4 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT


## 8. Acompanhando o loop no MLflow

Cada execução acima virou um trace no experimento `loop-agente-milho`. O `mlflow.langchain.autolog()` registra automaticamente cada chamada ao modelo e a cada ferramenta, em ordem. Cada volta do loop aparece como um grupo de spans.

Para abrir a interface:

1. Abra um terminal na pasta `notebooks/05-LoopEngineering` (o caminho do `mlflow.db` é relativo a ela).
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000`, escolha o experimento `loop-agente-milho` e abra a aba **Traces**.
4. Na árvore do trace, confira: o modelo decidindo, as ferramentas executando e a resposta final no fim.
5. Pressione `Ctrl+C` para parar a interface.

A célula seguinte lê os traces pelo próprio notebook.

In [8]:
mlflow.flush_trace_async_logging()  # garante que os traces já foram gravados

traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)
print(f"Traces registrados no experimento: {len(traces)}")

trace = mlflow.get_trace(traces.iloc[0]["trace_id"])
print(f"\nSpans do trace mais recente ({trace.info.trace_id}):")
for span in trace.data.spans:
    print(f"- {span.name} ({span.span_type})")

Traces registrados no experimento: 2

Spans do trace mais recente (tr-1e236498cc1395b7dab5f0172d921eae):
- LangGraph (CHAIN)
- PatchToolCallsMiddleware.before_agent (CHAIN)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- tools (CHAIN)
- tools (CHAIN)
- tools (CHAIN)
- tools (CHAIN)
- consultar_preco_milho (TOOL)
- consultar_preco_milho (TOOL)
- consultar_preco_milho (TOOL)
- tools (CHAIN)
- tools (CHAIN)
- consultar_preco_milho (TOOL)
- consultar_clima (TOOL)
- tools (CHAIN)
- consultar_clima (TOOL)
- tools (CHAIN)
- consultar_clima (TOOL)
- consultar_estoques (TOOL)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)


/tmp/ipykernel_20394/2430565787.py:3: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)


## Observações

- O loop é o mesmo em qualquer agente: decisão do modelo, ação na ferramenta, observação de volta ao modelo.
- Quem decide quando parar é o modelo, guiado pelo `system_prompt`. Por isso o prompt diz o que precisa ser coletado antes da conclusão.
- Os dados são simulados. Trocar os dicionários por chamadas reais de API não muda o loop, só as ferramentas.
- O `recursion_limit` é a proteção básica contra loops sem fim. Controles de custo e de qualidade ficam para os próximos notebooks.